PyTorch 的 Tensor（张量）是其核心数据结构，操作极其丰富。为了让你更系统地掌握，我将这些操作按功能分为了七大类：创建、属性、变形、索引切片、数学运算、设备管理以及保存加载。

以下是 PyTorch Tensor 的核心操作全览：
# 创建张量 (Creation)

这是使用 PyTorch 的第一步，可以通过数据、NumPy 数组或特定规则来创建。

- 从数据创建：
    - torch.tensor(data): 从 Python 列表或 NumPy 数组复制数据创建。
    - torch.from_numpy(ndarray): 直接从 NumPy 数组创建（共享内存，修改一个会影响另一个）。
- 初始化特定值：
    - torch.zeros(size): 创建全 0 张量。
    - torch.ones(size): 创建全 1 张量。
    - torch.full(size, value): 创建填充指定值的张量。
    - torch.eye(n): 创建单位矩阵。
        -  torch.diag(input, diagonal=0)
- 随机分布：
    - torch.rand(size): 均匀分布 (0, 1)。
    - torch.randn(size): 标准正态分布 (均值0, 方差1)。
    - torch.randint(low, high, size): 随机整数。
- 类似形状创建：
    - torch.zeros_like(input): 创建与 input 形状相同的全 0 张量。
    - torch.ones_like(input): 创建与 input 形状相同的全 1 张量。

## 从数据创建：
- torch.tensor(data): 从 Python 列表或 NumPy 数组复制数据创建。
- torch.from_numpy(ndarray): 直接从 NumPy 数组创建（共享内存，修改一个会影响另一个）

## 初始化特定值：
- torch.zeros(size): 创建全 0 张量。
- torch.ones(size): 创建全 1 张量。
- torch.full(size, value): 创建填充指定值的张量。
- torch.eye(n): 创建单位矩阵。

### torch.full(size, fill_value, *, dtype=None, device=None, requires_grad=False)
- size (序列): 定义输出张量的形状（例如 [2, 3] 或 (2, 3)）。
- fill_value (标量): 用来填充张量每一个元素的具体数值。
- dtype (可选): 数据类型（如 torch.int32, torch.float64）。如果不填，默认根据 fill_value 推断。
- device (可选): 设备（cpu 或 cuda）。
- requires_grad (可选): 是否需要计算梯度。


#### 创建特殊矩阵 (如掩码)
- 在深度学习中，经常需要创建全是 True 或 False 的掩码矩阵：

In [2]:
import torch
# 创建一个布尔类型的掩码
mask = torch.full((3,3),True,dtype=torch.bool)
print(mask)

tensor([[True, True, True],
        [True, True, True],
        [True, True, True]])


| 函数 | 等价于 | 用途 |
| :--- | :--- | :--- |
| `torch.zeros(size)` | `torch.full(size, 0)` | 创建全 0 张量 |
| `torch.ones(size)` | `torch.full(size, 1)` | 创建全 1 张量 |



#### torch.full_like()
- 它会根据输入张量 input 的形状来创建新张量：

In [ ]:
input_tensor = torch.randn(2,4) # 假设有一个2 * 4的随机张量
# 创建一个和 input_tensor 形状一样，但全是 -1 的张量
new_tensor = torch.full_like(input_tensor,-1)

### torch.eye(n, m=None, *, dtype=None, device=None, requires_grad=False)

torch.eye() 是 PyTorch 中专门用来生成单位矩阵的函数，它的核心作用就是创建一个二维张量，<font color='red'>让对角线上的元素为 1，其余位置全为 0。</font>虽然默认是方阵，但它也支持生成非方阵的“类单位矩阵”。
- n (int): 矩阵的行数。
- m (int, 可选): 矩阵的列数。如果不填，默认等于 n（即生成方阵）。
- dtype: 数据类型（默认通常是 torch.float32）。
- device: 设备（CPU 或 CUDA）。

#### 1. 创建标准方阵单位矩阵
- 只传入一个参数 n，创建一个 n×n 的矩阵。

In [ ]:
import torch

# 创建 3x3 的单位矩阵
identity = torch.eye(3)
print(identity)

# 输出:
# tensor([[1., 0., 0.],
#         [0., 1., 0.],
#         [0., 0., 1.]])

#### 创建非方阵 (矩形) 矩阵
- 传入 n 和 m，可以创建行数与列数不等的矩阵。对角线为 1，其余为 0。

In [ ]:
# 2 行 4 列
rectangular = torch.eye(2, 4)
print(rectangular)

# 输出:
# tensor([[1., 0., 0., 0.],
#         [0., 1., 0., 0.]])

#### 💡 常见应用场景
- 初始化权重: 在 RNN 或某些特殊网络层中，为了防止梯度消失或爆炸，有时会将权重矩阵初始化为单位矩阵（恒等初始化）。
- 提取对角线元素: 虽然 torch.diag() 更常用，但 eye 可以作为掩码（Mask）来提取或保留矩阵的对角线部分。
- 数学计算: 在线性代数运算中作为单位矩阵 I 使用（例如计算 A+I ）。

### torch.diag(input, diagonal=0)

torch.diag 这个函数很灵活，<font color='red'>具体做什么取决于你输入的是向量还是矩阵</font>，
主要分为两种用法:
- 构建对角矩阵
- 提取对角线元素

- input: 输入张量（可以是 1D 向量或 2D 矩阵）。
- diagonal: (可选) 指定对角线的位置。
    - 0: 主对角线 (默认)。
    - 正数: 主对角线上方。
    - 负数: 主对角线下方。

#### 🚀 两种主要用法

#####  输入是 1D 向量 (构建对角矩阵)

如果输入是一个 1D 张量，torch.diag 会创建一个 2D 方阵，将输入向量的元素放在对角线上，其余位置填充 0。

In [ ]:
import torch

x = torch.tensor([1, 2, 3])
result = torch.diag(x)

print(result)
# 输出:
# tensor([[1, 0, 0],
#         [0, 2, 0],
#         [0, 0, 3]])

##### 输入是 2D 矩阵 (提取对角线元素)

如果输入是一个 2D 张量，torch.diag 会提取出对角线上的元素，返回一个 1D 张量。

In [ ]:
matrix = torch.tensor([[1, 2, 3],
                       [4, 5, 6],
                       [7, 8, 9]])

# 提取主对角线元素
diag_elements = torch.diag(matrix)

print(diag_elements)
# 输出: tensor([1, 5, 9])

#### ⚙️ diagonal 参数的进阶用法
<font color='red'>你可以使用 diagonal 参数来操作非主对角线（上对角线或下对角线）。</font>

##### 在构建时 (输入是 1D 向量)
- diagonal > 0: 放在主对角线上方。
- diagonal < 0: 放在主对角线下方。

In [ ]:
x = torch.tensor([1, 2])

# 放在上方第一条对角线 (k=1)
print(torch.diag(x, diagonal=1))
# 输出:
# tensor([[0, 1, 0],
#         [0, 0, 2],
#         [0, 0, 0]])

# 放在下方第一条对角线 (k=-1)
print(torch.diag(x, diagonal=-1))
# 输出:
# tensor([[0, 0, 0],
#         [1, 0, 0],
#         [0, 2, 0]])

##### 在提取时 (输入是 2D 矩阵)

In [ ]:
matrix = torch.tensor([[1, 2, 3],
                       [4, 5, 6],
                       [7, 8, 9]])

# 提取上方第一条对角线
print(torch.diag(matrix, diagonal=1)) 
# 输出: tensor([2, 6])

# 提取下方第一条对角线
print(torch.diag(matrix, diagonal=-1)) 
# 输出: tensor([4, 8])

## 随机分布：

### torch.rand(*size, *, dtype=None, device=None, requires_grad=False)

torch.rand 是 PyTorch 中用于<font color='red'>生成服从均匀分布的随机数张量的函数</font>。它生成的数值会落在开区间 (0,1) 之间，是深度学习中最常用的初始化函数之一。

- size: 定义输出张量的形状。可以是一个元组（如 (2, 3)）或者一系列整数（如 2, 3）。
- dtype: 数据类型（默认通常是 torch.float32）。
- device: 设备（cpu 或 cuda）。

#### 创建指定形状的随机张量

生成一个 3x3 的矩阵，其中的数值都在 0 到 1 之间随机分布。

In [ ]:
import torch

# 创建一个 3x3 的随机张量
random_tensor = torch.rand(3, 3)
print(random_tensor)

# 输出示例 (每次运行结果都不同):
# tensor([[0.5488, 0.7152, 0.6028],
#         [0.5449, 0.4237, 0.6459],
#         [0.4376, 0.8918, 0.9637]])

####  生成任意范围 [a, b) 的随机数

- 公式：result=(b−a)×torch.rand(...)+a

##### 示例：生成 10 到 20 之间的随机数

In [ ]:
a = 10
b = 20

# 生成 5 个 10 到 20 之间的随机数
random_range = (b - a) * torch.rand(5) + a
print(random_range)
# 输出示例: tensor([13.24, 18.11, 10.56, 19.99, 15.43])

| 函数 | 分布类型 | 范围 | 典型用途 |
| :--- | :--- | :--- | :--- |
| `torch.rand` | 均匀分布 | (0, 1) | 权重初始化、Dropout 掩码 |
| `torch.randn` | 正态分布 |  $ (-\infty, +\infty) $  | 权重初始化 (高斯分布) |
| `torch.randint` | 整数均匀分布 |  $ [\text{low}, \text{high}) $  | 生成随机索引、数据增强 |

### torch.randn(*size, *, dtype=None, device=None, requires_grad=False)

torch.randn 是 PyTorch 中用于生成服从标准正态分布（Standard Normal Distribution）随机数的核心函数。

- *size: 定义输出张量的形状。可以是一系列整数（如 2, 3）或者一个元组（如 (2, 3)）。
- dtype: 数据类型（默认为 torch.float32）。
- device: 设备（cpu 或 cuda）。
- requires_grad: 是否需要计算梯度（默认为 False）。


#### 创建标准正态分布张量

生成一个 2x3 的矩阵，数值围绕 0 波动。

In [ ]:
import torch

# 创建一个 2x3 的随机张量
x = torch.randn(2, 3)
print(x)

# 输出示例 (每次运行结果不同):
# tensor([[ 0.5488, -0.7152,  0.6028],
#         [-0.5449,  0.4237, -1.6459]])

#### 调整分布：生成任意正态分布 $N(μ,σ^2)$


虽然 torch.randn 只能生成标准正态分布，但我们可以通过数学变换得到任意均值和标准差的分布。
- 公式：$X_{new}=X_{randn}×std+mean$

示例：生成均值为 5，标准差为 2 的随机数

In [ ]:
mean = 5
std = 2
# 先生成标准正态分布，再变换
custom_dist = torch.randn(2, 3) * std + mean
print(custom_dist)
# 数值将主要集中在 5 附近 (例如 3 到 7 之间)

####  💡 常见应用场景

| 场景 | 说明 |
| :--- | :--- |
| 神经网络权重初始化 | 最常用的场景。将卷积层或全连接层的权重初始化为标准正态分布，可以打破对称性，防止梯度消失或爆炸。 |
| 生成测试数据 | 模拟现实世界中带有噪声的输入数据（如图像像素的噪声）。 |
| Dropout | 虽然 Dropout 通常用伯努利分布，但在某些变体（如 Gaussian Dropout）中会用到正态分布噪声。 |
| 变分自编码器 | 在 VAE 的重参数化技巧中，用于从潜在空间中采样。 |



| 函数 | 分布类型 | 范围 | 典型用途 |
| :--- | :--- | :--- | :--- |
| `torch.randn` | 正态分布 (高斯) |  $ (-\infty, +\infty) $ <br>(集中在 0 附近) | 权重初始化、噪声注入 |
| `torch.rand` | 均匀分布 |  $ (0, 1) $  | Dropout 掩码、概率采样 |
